In [1]:
# use chat gpt suggestions

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models

In [3]:
# Transformations for data augmentation and normalization
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])

# Load CIFAR-10 dataset
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

# Data loaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=4)


Files already downloaded and verified
Files already downloaded and verified


In [4]:
#load a pretrained model
# Load ResNet-18 pre-trained on ImageNet
model = models.resnet18(pretrained=True)

# Modify the final layer for CIFAR-10 (10 classes)
num_ftrs = model.fc.in_features  # Get the number of input features to the last layer
model.fc = nn.Linear(num_ftrs, 10)  # Replace it with a new fully connected layer for 10 classes


/home/iulian/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/iulian/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


In [6]:
criterion = nn.CrossEntropyLoss()  # Cross-entropy for classification
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)  # SGD optimizer
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)  # Cosine annealing learning rate scheduler


In [7]:
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()  # Zero the gradients
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        running_loss += loss.item()
    scheduler.step()  # Update the learning rate

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}")
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    print(f"Test Accuracy: {100 * correct / total:.2f}%")


Epoch 1/20, Loss: 2.9676
Test Accuracy: 23.74%
Epoch 2/20, Loss: 1.9251
Test Accuracy: 31.24%
Epoch 3/20, Loss: 1.7011
Test Accuracy: 40.95%
Epoch 4/20, Loss: 1.5730
Test Accuracy: 45.27%
Epoch 5/20, Loss: 1.4562
Test Accuracy: 44.85%
Epoch 6/20, Loss: 1.3441
Test Accuracy: 44.97%
Epoch 7/20, Loss: 1.2511
Test Accuracy: 59.19%
Epoch 8/20, Loss: 1.1712
Test Accuracy: 52.77%
Epoch 9/20, Loss: 1.1172
Test Accuracy: 61.07%
Epoch 10/20, Loss: 1.0637
Test Accuracy: 64.19%
Epoch 11/20, Loss: 1.0264
Test Accuracy: 61.68%
Epoch 12/20, Loss: 0.9968
Test Accuracy: 66.41%
Epoch 13/20, Loss: 0.9671
Test Accuracy: 61.03%
Epoch 14/20, Loss: 0.9367
Test Accuracy: 67.10%
Epoch 15/20, Loss: 0.9121
Test Accuracy: 66.79%
Epoch 16/20, Loss: 0.8837
Test Accuracy: 66.91%
Epoch 17/20, Loss: 0.8757
Test Accuracy: 67.98%
Epoch 18/20, Loss: 0.8549
Test Accuracy: 65.17%
Epoch 19/20, Loss: 0.8380
Test Accuracy: 68.76%
Epoch 20/20, Loss: 0.8301
Test Accuracy: 72.00%


In [8]:
torch.save(model.state_dict(), "resnet18_cifar10.pth")

In [9]:
# if we need to load the model later
model.load_state_dict(torch.load("resnet18_cifar10.pth"))
model = model.to(device)


/tmp/ipykernel_16781/3932687741.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("resnet18_cifar10.pth"))
